TEXT SUMMARIZER :
steps:
1. load dataset
2. random sampling
3. preprocessing
4. Tokenization
5. load model (T5Tokenizer)
6. fine tuning(define arguments
trainer values)
7. training model
8. save and load the model
9. testing...

In [ ]:
import numpy as np
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

1. Load Dataset

In [ ]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [ ]:
train_data.head()

In [ ]:
train_data["dialogue"][0]

In [ ]:
train_data.sample(5)

In [ ]:
train_data.shape, val_data.shape

2. Ramdom Sampling

In [ ]:
#random sampling(training only on few data).

train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [ ]:
train_data.shape, val_data.shape

3. Data Preprossesing

In [ ]:
import re

def clean_data(text):
  text = re.sub(r"\r\n", " ", text) #remove lines
  text = re.sub(r"\s+", " ", text) #remove spaces
  text = re.sub(r"<.*?>", " ", text)
  text.strip().lower()
  return text

In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [ ]:
train_data["dialogue"][0]

4. Tokenization

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [ ]:
# raw data => tokenized inputs for fine-tuning

def tokenize(data):
  inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
  target = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True) #150 cuz summary smaller than dialogue

  inputs["labels"] = target["input_ids"]
  return inputs

In [ ]:
# inputs["labels"] = target["input_ids"].
# This means the model will use the dialogue tokens as input and the summary tokens as the expected output (labels) during training.

In [ ]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [ ]:
train_dataset[0]

In [ ]:
# input ids : token ids of dialogue
# 1 => EOS, 0 => padding

# attention mask : tells which values are valid, which values are padding vaues.

# label : token ids of summary

In [ ]:
len(train_dataset[0]["input_ids"])

In [ ]:
type(train_dataset)
type(val_dataset)

6. Working with our model (Load Model)

In [ ]:
#NLP => generation task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

7. Defining Training Argumments

In [ ]:
# defining training argumments

training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs=4,
    eval_strategy="epoch",
    save_strategy="epoch",
)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

8. Training

In [ ]:
trainer.train()

In [ ]:
# load model => fine-tune => save model.

Save model

In [ ]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model") #we have to tokenize user input.

load model

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model") #we have to tokenize user input.

In [ ]:
def summarize_dialogue(dialogue):

  #clean
  dialogue = clean_data(dialogue)

  #tokenize
  inputs = tokenizer(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt" #returns pytorch tensors
  ).to(device)

  #generate
  model.to(device)
  targets = model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length=150,
      num_beams=4, #transformer will generate 4 different types of summaries and returns the best summary.
      early_stopping=True
  )

  # token ids convert to summary => decoding.

  #decoded our output.
  summary = tokenizer.decode(targets[0], skip_special_tokens=True) # special_tokens : SEP, EOS, PADDING
  return summary

In [ ]:
test_dialogue = """
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye
"""

summary = summarize_dialogue(test_dialogue)

print("Summary : ", summary)